# 11. Conditional blank-line rule

Owner: rdevasia25. Transferred from Gtpatel02. Issue #11. Depends on completed #10.

## Goal

Compare the fixed eight-candidate formatting grid on the same five training folds as #10. Keep soft offset 2 as the primary path and hard offset 1000 as a separate ablation.

## Setup

A blank line can be a dataset shortcut. Activate its rule only if the current original fit partition has zero human examples with literal two consecutive newlines, before any specialist filtering.

In [ ]:
import os, json, time, warnings, gc, traceback
from pathlib import Path
import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython import get_ipython
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
FORMAT_HELPERS_ONLY = os.environ.pop("FORMATTING_RULE_HELPERS_ONLY", "0") == "1"
os.environ["EXPANDED_TEXT_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./10_expanded_text.ipynb")
FORMATTING_GRID = [
    {"name":"reference", "view":"word_char", "learner":"lr", "C":16., "fit_mode":"all", "rule":False},
    {"name":"rule_all_wordchar16", "view":"word_char", "learner":"lr", "C":16., "fit_mode":"all", "rule":True},
    {"name":"rule_no_blank_wordchar16", "view":"word_char", "learner":"lr", "C":16., "fit_mode":"no_blank", "rule":True},
    {"name":"rule_no_blank_wordchar64", "view":"word_char", "learner":"lr", "C":64., "fit_mode":"no_blank", "rule":True},
    {"name":"rule_no_blank_casechar64", "view":"casechar", "learner":"lr", "C":64., "fit_mode":"no_blank", "rule":True},
    {"name":"rule_no_blank_case_svm1", "view":"casechar", "learner":"svm", "C":1., "fit_mode":"no_blank", "rule":True},
    {"name":"rule_no_blank_nbword4", "view":"nbword13", "learner":"lr", "C":4., "fit_mode":"no_blank", "rule":True},
    {"name":"rule_no_blank_word13_16", "view":"word13", "learner":"lr", "C":16., "fit_mode":"no_blank", "rule":True},
]
FORMATTING_OFFSETS = {"soft":2., "hard":1000.}


## Checks

Notebook 02 verifies data bytes, duplicate audit, and fixed split. Preserve raw text, including embedded carriage returns. The rule uses literal `\n\n`, not the broader whitespace-blank regex from #13. Features and NB ratios fit on training rows only. Margin reference statistics use the explicitly allowed eligible unlabeled rows, with blank-line rows excluded for specialists.

In [ ]:
if not FORMAT_HELPERS_ONLY:
    NAME = "11_formatting_rule"
    COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=3600 notebooks/11_formatting_rule.ipynb"
    START = time.perf_counter()
    parts = load_partitions()
    train, validation, reference = parts["train"], parts["validation"], parts["unlabeled"]
    folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train.text, train.label))
    records = []
    for path in sorted((ROOT/"results/10_expanded_text").glob("*.json")):
        record = json.loads(path.read_text())
        if record.get("status") == "completed": records.append((path, record))
    require(bool(records), "Completed #10 is required")
    dependency_path, dependency = records[-1]
    source_hash = value_hash(["".join(c["source"]) for c in json.loads((ROOT/"notebooks/10_expanded_text.ipynb").read_text())["cells"]])
    require(source_hash == dependency["config"]["notebook_source_sha256"], "Notebook 10 source changed since its recorded run")
    fold_hashes = [{"fit":value_hash(train.id.iloc[a].tolist()), "score":value_hash(train.id.iloc[b].tolist())} for a,b in folds]
    require(fold_hashes == dependency["config"]["fold_id_sha256"], "Folds differ from #10")
    CONFIG = {"grid":FORMATTING_GRID, "offsets":FORMATTING_OFFSETS, "primary":"soft", "cv":{"folds":5,"shuffle":True,"seed":42,"metric":"pooled OOF AUROC"},
        "fold_id_sha256":fold_hashes, "views":EXPANDED_VIEWS, "learners":EXPANDED_LEARNERS, "threads":1,
        "tfidf":dependency["config"]["tfidf"], "nb":dependency["config"]["nb"],
        "rule":"literal two consecutive newlines; zero human matches in original current fit partition before specialist filtering",
        "reference":"eligible unlabeled; no-blank specialists exclude literal blank lines; store population mean/std; never normalize test batches",
        "ensemble":{"min_gain":.0005,"max_members":5,"repeat_members":False,"tie":"grid order; fewer within .0005 of best path"},
        "hard_probability_mapping":"fixed arctan mapping of the pre-offset ensemble margin: flagged [.99,1), others (0,.99); check observed score order without labels",
        "soft_probability_mapping":"sigmoid of the averaged offset margins; not claimed calibrated",
        "reuse":"reference and rule-all share identical learned fit/reference stats; only rule score differs",
        "dependency":{"record":str(dependency_path.relative_to(ROOT)),"record_sha256":file_hash(dependency_path),"source_sha256":source_hash},
        "notebook_source_sha256":value_hash(["".join(c["source"]) for c in json.loads((ROOT/"notebooks/11_formatting_rule.ipynb").read_text())["cells"]])}
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    private = ROOT/"artifacts"/NAME/RUN_ID
    private.mkdir(parents=True,exist_ok=False)


## Method

Use the exact #10 text settings. Specialists remove blank-line rows from their fit data and margin reference, but their rule eligibility is checked on the original fit rows. This prevents a human counterexample from disappearing through filtering. No evaluation label activates a rule.

Run the same greedy selection separately for soft and hard scores, then save both choices before validation. Soft is primary by design. For the hard ablation, use a fixed monotone arctan mapping instead of applying sigmoid to an offset near 1000. Do not rank-normalize any evaluation batch. Check probability ranges and score order on every saved partition.

In [ ]:
def formatting_flags(texts):
    return np.asarray(["\n\n" in str(text) for text in texts], dtype=bool)

def hard_format_probabilities(base, flags, scores):
    base, flags, scores = np.asarray(base,dtype=float), np.asarray(flags,dtype=bool), np.asarray(scores,dtype=float)
    require(base.shape == flags.shape == scores.shape and np.isfinite(base).all() and np.isfinite(scores).all(), "Invalid hard mapping input")
    unit = .5 + np.arctan(base) / np.pi
    probabilities = np.where(flags, .99 + .01*unit, .99*unit)
    probabilities = np.where(flags, np.clip(probabilities,.99,np.nextafter(1.,0.)),
                              np.clip(probabilities,np.nextafter(0.,1.),np.nextafter(.99,0.)))
    require(((probabilities[flags] >= .99) & (probabilities[flags] < 1.)).all(), "Flagged hard range failed")
    require(((probabilities[~flags] > 0.) & (probabilities[~flags] < .99)).all(), "Unflagged hard range failed")
    order = np.argsort(scores,kind="stable")
    differences, probability_differences = np.diff(scores[order]), np.diff(probabilities[order])
    require((probability_differences[differences > 0] > 0).all(), "Hard mapping changed score order or lost precision")
    require((probability_differences[differences == 0] == 0).all(), "Hard mapping broke a score tie")
    return probabilities

class FormattingArm:
    classes_ = np.array([0,1])
    def __init__(self,candidate): self.candidate = dict(candidate)
    def fit(self,fit_frame,reference_frame,shared=None):
        texts, y = fit_frame.text, fit_frame.label.to_numpy()
        literal = formatting_flags(texts)
        require(set(y) == {0,1}, "Both original fit classes are required")
        human_blank = int(((y == 0) & literal).sum())
        self.rule_active_ = bool(self.candidate["rule"] and human_blank == 0)
        self.fit_signature_ = value_hash({"ids":fit_frame.id.tolist(),"labels":y.tolist(),"texts":texts.tolist()})
        keep = ~literal if self.candidate["fit_mode"] == "no_blank" else np.ones(len(texts),dtype=bool)
        reference_keep = ~formatting_flags(reference_frame.text) if self.candidate["fit_mode"] == "no_blank" else np.ones(len(reference_frame),dtype=bool)
        require(set(y[keep]) == {0,1} and reference_keep.any(), "Specialist fit/reference is degenerate")
        self.audit_ = {"original_fit_rows":len(fit_frame),"used_fit_rows":int(keep.sum()),
            "used_human":int((y[keep] == 0).sum()),"used_AI":int((y[keep] == 1).sum()),
            "human_literal_blank_before_filter":human_blank,"original_literal_blank":int(literal.sum()),
            "rule_active":self.rule_active_,"reference_rows":int(reference_keep.sum()),
            "fit_ids_sha256":value_hash(fit_frame.id.iloc[np.flatnonzero(keep)].tolist()),
            "reference_ids_sha256":value_hash(reference_frame.id.iloc[np.flatnonzero(reference_keep)].tolist())}
        if shared is not None:
            require(self.fit_signature_ == shared.fit_signature_, "Reused fit data differs")
            require(all(self.candidate[k] == shared.candidate[k] for k in ["view","learner","C","fit_mode"]), "Reused model settings differ")
            self.model_, self.statistics_ = shared.model_, dict(shared.statistics_)
            self.fit_seconds_, self.reference_seconds_, self.reused_ = 0., 0., True
        else:
            start = time.perf_counter()
            self.model_ = make_expanded_model(self.candidate).fit(texts.iloc[np.flatnonzero(keep)],y[keep])
            self.fit_seconds_ = time.perf_counter()-start
            start = time.perf_counter()
            self.statistics_ = margin_reference(self.model_,reference_frame.text.iloc[np.flatnonzero(reference_keep)])
            self.reference_seconds_, self.reused_ = time.perf_counter()-start, False
        return self
    def components(self,texts):
        raw = np.asarray(self.model_.decision_function(texts),dtype=float)
        require(np.isfinite(raw).all(), "Invalid raw margin")
        standardized = (raw-self.statistics_["mean"])/self.statistics_["std"]
        flags = formatting_flags(texts) & self.rule_active_
        return raw, standardized, flags
    def decision_function(self,texts,offset=2.):
        _, margin, flags = self.components(texts)
        return margin + offset*flags
    def predict_proba(self,texts):
        p = expit(self.decision_function(texts))
        return np.column_stack([1-p,p])

class FormattingEnsemble:
    classes_ = np.array([0,1])
    def __init__(self,arms,offset=2.): self.arms, self.offset = arms, float(offset)
    def components(self,texts):
        values = [arm.components(texts) for arm in self.arms]
        base = np.mean([v[1] for v in values],axis=0)
        flag_weight = np.mean([v[2].astype(float) for v in values],axis=0)
        return base, flag_weight
    def decision_function(self,texts,offset=None):
        base, flag_weight = self.components(texts)
        return base + (self.offset if offset is None else offset)*flag_weight
    def predict_proba(self,texts):
        base, flag_weight = self.components(texts)
        scores = base+self.offset*flag_weight
        p = hard_format_probabilities(base,flag_weight > 0,scores) if self.offset == 1000. else expit(scores)
        return np.column_stack([1-p,p])

def formatting_probability_frame(model,frame):
    return pd.DataFrame({"id":frame.id.to_numpy(),"p_ai":class1_probabilities(model,frame.text)})


## Run

Commit the grid before fitting. Keep every raw margin, standardized margin, literal flag, and active flag under ignored artifacts. Save fold-specific rule state and reference statistics. Progress and detailed failures stay private; safe run records keep only error types and aggregates.

In [ ]:
if not FORMAT_HELPERS_ONLY:
    plan_path = ROOT/"results"/NAME/f"{RUN_ID}-plan.json"
    save_new_json(plan_path,{"config":CONFIG,"config_sha256":value_hash(CONFIG),"saved_before_fit":True,"split_sha256":parts["split"]["split_sha256"]})
    artifacts, fold_rows, candidate_rows = [plan_path], [], []
    def progress(stage,**values):
        with (private/"progress.jsonl").open("a") as handle:
            handle.write(json.dumps({"utc":datetime.now(timezone.utc).isoformat(),"stage":stage,**values})+"\n")
    try:
        names = [c["name"] for c in FORMATTING_GRID]
        oof_base = {name:np.full(len(train),np.nan) for name in names}
        oof_raw = {name:np.full(len(train),np.nan) for name in names}
        oof_flags = {name:np.zeros(len(train),dtype=bool) for name in names}
        oof_fold = np.full(len(train),-1)
        search_start = time.perf_counter()
        with threadpool_limits(limits=1),warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            for fold,(a,b) in enumerate(folds):
                original_fit = train.iloc[a]
                shared = None
                oof_fold[b] = fold
                for candidate in FORMATTING_GRID:
                    start = time.perf_counter()
                    arm = FormattingArm(candidate).fit(original_fit,reference,shared=shared if candidate["name"] == "rule_all_wordchar16" else None)
                    raw, margin, flags = arm.components(train.text.iloc[b])
                    name = candidate["name"]
                    oof_raw[name][b], oof_base[name][b], oof_flags[name][b] = raw, margin, flags
                    row = {"candidate":name,"fold":fold,"fit_seconds":arm.fit_seconds_,"reference_seconds":arm.reference_seconds_,
                        "total_seconds":time.perf_counter()-start,"reused_base_fit":arm.reused_,"reference":arm.statistics_,"audit":arm.audit_}
                    row.update({f"{mode}_auroc":float(roc_auc_score(train.label.iloc[b],margin+offset*flags)) for mode,offset in FORMATTING_OFFSETS.items()})
                    fold_rows.append(row)
                    progress("cv_fold",candidate=name,fold=fold,seconds=row["total_seconds"])
                    if name == "reference": shared = arm
                    else: del arm
                    gc.collect()
                del shared
        search_seconds = time.perf_counter()-search_start
        score_sets = {mode:{name:oof_base[name]+offset*oof_flags[name] for name in names} for mode,offset in FORMATTING_OFFSETS.items()}
        selections = {}
        for name in names:
            require(np.isfinite(oof_base[name]).all() and (oof_fold >= 0).all(),"Incomplete OOF coverage")
            hard_p = hard_format_probabilities(oof_base[name],oof_flags[name],score_sets["hard"][name])
            row = {"candidate":name}
            for mode in FORMATTING_OFFSETS:
                scores = [r[f"{mode}_auroc"] for r in fold_rows if r["candidate"] == name]
                row.update({f"{mode}_pooled_oof":float(roc_auc_score(train.label,score_sets[mode][name])),
                            f"{mode}_fold_mean":float(np.mean(scores)),f"{mode}_fold_std":float(np.std(scores))})
            candidate_rows.append(row)
            path = private/f"{name}_oof.csv"
            pd.DataFrame({"id":train.id,"fold":oof_fold,"raw_margin":oof_raw[name],"standardized_margin":oof_base[name],
                "literal_blank":formatting_flags(train.text),"active_flag":oof_flags[name],
                "soft_score":score_sets["soft"][name],"hard_score":score_sets["hard"][name],
                "soft_p_ai":expit(score_sets["soft"][name]),"hard_p_ai":hard_p}).to_csv(path,index=False)
            artifacts.append(path)
        prior_reference = next(r for r in dependency["metrics"]["candidate_results"] if r["candidate"] == "word13_casechar_lr16")
        reference_oof_difference = float(candidate_rows[0]["soft_pooled_oof"]-prior_reference["pooled_oof_auroc"])
        require(abs(reference_oof_difference) <= 1e-9,"Reference OOF no longer matches #10")
        for mode in FORMATTING_OFFSETS:
            selections[mode] = greedy_margin_selection(score_sets[mode],train.label,names,min_gain=.0005,max_members=5)
            members = selections[mode]["selected"]
            base = np.mean([oof_base[name] for name in members],axis=0)
            flag_weight = np.mean([oof_flags[name].astype(float) for name in members],axis=0)
            scores = base+FORMATTING_OFFSETS[mode]*flag_weight
            p = hard_format_probabilities(base,flag_weight > 0,scores) if mode == "hard" else expit(scores)
            require(abs(float(roc_auc_score(train.label,p))-selections[mode]["selected_oof_auroc"]) <= 1e-12,"Probability mapping changed OOF AUROC")
            path = private/f"{mode}_ensemble_oof.csv"
            pd.DataFrame({"id":train.id,"fold":oof_fold,"base_margin":base,"active_flag_weight":flag_weight,"score":scores,"p_ai":p}).to_csv(path,index=False);artifacts.append(path)
        selection_path = ROOT/"results"/NAME/f"{RUN_ID}-selection.json"
        save_new_json(selection_path,{"selections":selections,"candidate_results":candidate_rows,"validation_inspected":False,
                                     "utc":datetime.now(timezone.utc).isoformat(),"config_sha256":value_hash(CONFIG)})
        artifacts.append(selection_path)
        needed = set(selections["soft"]["selected"]) | set(selections["hard"]["selected"])
        retained, saved_candidates, final_rows = {}, {}, []
        with threadpool_limits(limits=1),warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            shared = None
            for candidate in FORMATTING_GRID:
                name = candidate["name"]
                arm = FormattingArm(candidate).fit(train,reference,shared=shared if name == "rule_all_wordchar16" else None)
                row = {"candidate":name,"fit_seconds":arm.fit_seconds_,"reference_seconds":arm.reference_seconds_,
                       "reused_base_fit":arm.reused_,"reference":arm.statistics_,"audit":arm.audit_}
                for partition,frame in [("train",train),("validation",validation)]:
                    raw,base,flags = arm.components(frame.text)
                    soft_score, hard_score = base+2.*flags,base+1000.*flags
                    soft_p, hard_p = expit(soft_score),hard_format_probabilities(base,flags,hard_score)
                    row[f"{partition}_soft_auroc"] = float(roc_auc_score(frame.label,soft_p))
                    row[f"{partition}_hard_auroc"] = float(roc_auc_score(frame.label,hard_p))
                    path = private/f"{name}_{partition}.csv"
                    pd.DataFrame({"id":frame.id,"raw_margin":raw,"standardized_margin":base,"literal_blank":formatting_flags(frame.text),
                        "active_flag":flags,"soft_score":soft_score,"hard_score":hard_score,"soft_p_ai":soft_p,"hard_p_ai":hard_p}).to_csv(path,index=False);artifacts.append(path)
                path = private/f"{name}.pkl"
                with path.open("wb") as handle: cloudpickle.dump(arm,handle)
                artifacts.append(path)
                saved_candidates[name] = {"model":str(path.relative_to(ROOT)),"config":candidate,"reference":arm.statistics_,"audit":arm.audit_}
                final_rows.append(row);progress("full_fit",candidate=name)
                if name in needed: retained[name] = arm
                if name == "reference": shared = arm
                else: del arm
                gc.collect()
            del shared
        prior_final = next(r for r in dependency["metrics"]["final_results"] if r["candidate"] == "word13_casechar_lr16")
        reference_validation_difference = float(final_rows[0]["validation_soft_auroc"]-prior_final["validation_auroc"])
        require(abs(reference_validation_difference) <= 1e-9,"Reference validation differs from #10")
        ensemble_rows, saved_ensembles, subgroup_rows = [], {}, []
        with threadpool_limits(limits=1):
            for mode,offset in FORMATTING_OFFSETS.items():
                members = selections[mode]["selected"]
                model = FormattingEnsemble([retained[name] for name in members],offset=offset)
                path = private/f"{mode}_ensemble.pkl"
                with path.open("wb") as handle: cloudpickle.dump(model,handle)
                artifacts.append(path);saved_ensembles[mode] = str(path.relative_to(ROOT))
                row = {"mode":mode,"members":members,"offset":offset,"pooled_oof_auroc":selections[mode]["selected_oof_auroc"]}
                for partition,frame in [("train",train),("validation",validation)]:
                    predictions = formatting_probability_frame(model,frame)
                    metrics = evaluate_predictions(frame,predictions)
                    row[f"{partition}_auroc"],row[f"{partition}_accuracy_at_0_5"] = metrics["auroc"],metrics["accuracy_at_0_5"]
                    base,flag_weight = model.components(frame.text)
                    path = private/f"{mode}_ensemble_{partition}.csv"
                    predictions.assign(base_margin=base,active_flag_weight=flag_weight,score=base+offset*flag_weight).to_csv(path,index=False);artifacts.append(path)
                    if partition == "validation":
                        for group,mask in [("literal_blank",formatting_flags(frame.text)),("no_literal_blank",~formatting_flags(frame.text))]:
                            part = frame.iloc[np.flatnonzero(mask)]
                            result = evaluate_predictions(part,predictions.iloc[np.flatnonzero(mask)]) if len(part) else {"auroc":None,"accuracy_at_0_5":None,"rows":0}
                            subgroup_rows.append({"mode":mode,"group":group,"human":int((part.label == 0).sum()),"AI":int((part.label == 1).sum()),**result})
                with (private/f"{mode}_ensemble.pkl").open("rb") as handle: restored = cloudpickle.load(handle)
                expected = pd.read_csv(private/f"{mode}_ensemble_validation.csv",dtype={"id":"string"},float_precision="round_trip")
                require(expected.id.tolist() == validation.id.tolist(),"Reload IDs changed")
                difference = float(np.max(np.abs(class1_probabilities(restored,validation.text)-expected.p_ai.to_numpy())))
                require(difference <= 1e-12,"Reload changed ensemble probabilities")
                row["max_reload_probability_difference"] = difference
                ensemble_rows.append(row)
        spec_path = private/"refit_spec.json"
        save_new_json(spec_path,{"config":CONFIG,"selected_soft":selections["soft"]["selected"],"selected_hard":selections["hard"]["selected"],
            "candidates":saved_candidates,"fit_order_ids_sha256":value_hash(train.id.tolist()),
            "rule_condition_evaluated_before_specialist_filter":True})
        artifacts.append(spec_path)
    except Exception as exc:
        (private/"failure_traceback.txt").write_text(traceback.format_exc())
        record_run(NAME,COMMAND,CONFIG,SEED,len(train),len(validation),{"validation_auroc":None,"completed_folds":fold_rows},
                   time.perf_counter()-START,status="failed",error_type=type(exc).__name__)
        raise RuntimeError("Formatting experiment failed. See safe record and ignored failure_traceback.txt.") from None


## Results

Soft and hard select on training OOF only. Neither can use the validation table to change its members. The hard probability bands express the rule, not measured confidence calibration. A training formatting pattern may fail on other data; these results do not establish hidden-test subgroup behavior.

In [ ]:
if not FORMAT_HELPERS_ONLY:
    display(pd.DataFrame(candidate_rows))
    display(pd.DataFrame([{k:v for k,v in row.items() if k not in ["audit","reference"]} for row in final_rows]))
    display(pd.DataFrame(ensemble_rows));display(pd.DataFrame(subgroup_rows))
    display(pd.DataFrame([{"candidate":r["candidate"],**r["audit"]} for r in final_rows]))
    print(f"Search {search_seconds:.3f}s; #10 reference OOF difference {reference_oof_difference:.3g}; validation difference {reference_validation_difference:.3g}")
    fig,axes = plt.subplots(1,2,figsize=(12,5))
    x = np.arange(len(candidate_rows))
    for mode,offset in [("soft",-.18),("hard",.18)]:
        axes[0].barh(x+offset,[r[f"{mode}_pooled_oof"] for r in candidate_rows],height=.36,label=mode)
    axes[0].set(yticks=x,yticklabels=[r["candidate"] for r in candidate_rows],xlim=(0,1),xlabel="Training OOF AUROC",title="Fixed formatting grid")
    axes[0].tick_params(axis="y",labelsize=8);axes[0].legend()
    for mode,offset in [("soft",-.18),("hard",.18)]:
        axes[1].bar(np.arange(2)+offset,[next(r for r in ensemble_rows if r["mode"] == mode)[f"{p}_auroc"] for p in ["train","validation"]],width=.36,label=mode)
    axes[1].set(xticks=np.arange(2),xticklabels=["train","validation"],ylim=(0,1),ylabel="AUROC",title="Frozen CV choices");axes[1].legend()
    fig.tight_layout();plot_path = private/"comparison.png";fig.savefig(plot_path,dpi=140);artifacts.append(plot_path);plt.show()


## Takeaway

- Tried the eight fixed candidates with training-only rule checks, then separate soft and hard greedy selections.
- Tables keep the reference, losing arms, fold variation, rule state, and held-out confirmation. Hard bands are a shortcut ablation, not calibrated authorship probabilities.
- Next: #12 tests split, solver, and formatting sensitivity. Keep fit rules and reference statistics frozen during evaluation.

## Save

`artifacts/11_formatting_rule/latest.json` exposes both portable ensembles, all candidate settings, ID-aligned scores, and the standalone refit specification. Raw margins and flags remain ignored. For #12 reuse, set `FORMATTING_RULE_HELPERS_ONLY=1` before `%run ./11_formatting_rule.ipynb`; this defines factories without fitting or writing.

In [ ]:
if not FORMAT_HELPERS_ONLY:
    CONFIG["extra_versions"] = {name:importlib.metadata.version(name) for name in ["cloudpickle","matplotlib","threadpoolctl"]}
    metrics = {"validation_auroc":next(r["validation_auroc"] for r in ensemble_rows if r["mode"] == "soft"),
        "primary":"soft","candidate_results":candidate_rows,"fold_results":fold_rows,"final_results":final_rows,
        "ensemble_results":ensemble_rows,"validation_subgroups":subgroup_rows,"selections":selections,
        "search_seconds":search_seconds,"reference_oof_difference":reference_oof_difference,"reference_validation_difference":reference_validation_difference}
    record_path = record_run(NAME,COMMAND,CONFIG,SEED,len(train),len(validation),metrics,time.perf_counter()-START,artifacts=artifacts)
    pointer = {"result":str(record_path.relative_to(ROOT)),"result_sha256":file_hash(record_path),"split_sha256":parts["split"]["split_sha256"],
        "model":saved_ensembles["soft"],"hard_model":saved_ensembles["hard"],"primary":"soft",
        "selected_soft":selections["soft"]["selected"],"selected_hard":selections["hard"]["selected"],"candidates":saved_candidates,
        "refit_spec":str(spec_path.relative_to(ROOT)),"train_predictions":str((private/"soft_ensemble_train.csv").relative_to(ROOT)),
        "validation_predictions":str((private/"soft_ensemble_validation.csv").relative_to(ROOT)),
        "hard_validation_predictions":str((private/"hard_ensemble_validation.csv").relative_to(ROOT)),
        "artifacts":{str(path.relative_to(ROOT)):file_hash(path) for path in artifacts}}
    target = ROOT/"artifacts"/NAME/"latest.json";temporary = target.with_suffix(".tmp")
    temporary.write_text(json.dumps(pointer,sort_keys=True,indent=2)+"\n");temporary.replace(target)
    progress("completed",result=str(record_path.relative_to(ROOT)))
    print("Saved conditional-rule comparison and both portable handoffs.")
